# 🎲 Probability & Distributions for ML
### Part of the *Math for ML* section — ML-CaPsule

> **Goal:** Understand Bayes' theorem, conditional probability, and probability distributions — then connect them directly to the **Naive Bayes** implementation already in this repo.

---

## 🗺️ What you'll learn

| Concept | ML Connection |
|---|---|
| Basic probability | Priors and class probabilities in classifiers |
| Conditional probability | P(spam \| word) — the core of Naive Bayes |
| Bayes' Theorem | The formula that powers the Naive Bayes classifier |
| Normal distribution | Gaussian Naive Bayes — used when features are continuous |
| PDF and CDF | How likelihoods are computed for continuous features |

---

## 📺 Watch first (optional but highly recommended)
[StatQuest with Josh Starmer — Naive Bayes playlist](https://www.youtube.com/watch?v=O2L2Uv9pdDA)  
[StatQuest — Probability Fundamentals](https://www.youtube.com/watch?v=uzkc-qNVoOk)

---
## 1️⃣ Basic Probability

### Concept
The **probability** of an event $A$ is a number between 0 and 1:

$$P(A) = \frac{\text{number of outcomes where A occurs}}{\text{total number of outcomes}}$$

**Key rules:**
- $0 \leq P(A) \leq 1$
- $P(A) + P(\text{not } A) = 1$
- $P(A \text{ or } B) = P(A) + P(B) - P(A \text{ and } B)$

**ML connection:** In a classification problem, $P(\text{class})$ is the **prior** — how likely each class is before seeing any features.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Email dataset: 1000 emails
total_emails = 1000
spam_emails  = 300
ham_emails   = 700

P_spam = spam_emails / total_emails
P_ham  = ham_emails  / total_emails

print("=== Prior Probabilities (before seeing any email content) ===")
print(f"P(spam) = {spam_emails}/{total_emails} = {P_spam:.2f}")
print(f"P(ham)  = {ham_emails}/{total_emails}  = {P_ham:.2f}")
print(f"Sum     = {P_spam + P_ham:.2f}  ✅ (must equal 1)")
print()
print("This is the 'prior' — what we know BEFORE reading the email.")
print("A Naive Bayes classifier uses this as its starting belief.")

In [ ]:
# Visualise class distribution
fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(['Spam', 'Ham (not spam)'], [P_spam, P_ham], color=['tomato', 'steelblue'], width=0.5)
for bar, val in zip(bars, [P_spam, P_ham]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{val:.2f}', ha='center', fontsize=12)
ax.set_ylim(0, 1.0)
ax.set_ylabel('Probability')
ax.set_title('Prior class probabilities P(class)')
ax.grid(axis='y', alpha=0.4)
plt.tight_layout(); plt.show()

---
## 2️⃣ Conditional Probability

### Concept
**Conditional probability** $P(A | B)$ is the probability of event $A$ **given that** $B$ has already occurred:

$$P(A \mid B) = \frac{P(A \text{ and } B)}{P(B)}$$

**ML connection:** In spam detection:
$$P(\text{"free" in email} \mid \text{spam}) = \frac{\text{spam emails containing "free"}}{\text{total spam emails}}$$

This is the **likelihood** — how probable is this feature given the class?

In [ ]:
# Conditional probabilities: how often do certain words appear in spam vs ham?
# P(word | class)

# Counts: how many spam/ham emails contain each word
word_counts = {
    'free':   {'spam': 240, 'ham':  70},   # 'free' appears a lot in spam
    'meeting':{'spam':  30, 'ham': 350},   # 'meeting' appears a lot in ham
    'the':    {'spam': 270, 'ham': 650},   # 'the' appears everywhere
}

print("Conditional probabilities P(word | class):\n")
print(f"{'Word':<10} {'P(word|spam)':<18} {'P(word|ham)':<18} {'Spam indicator?'}")
print("-" * 65)

for word, counts in word_counts.items():
    p_word_given_spam = counts['spam'] / spam_emails
    p_word_given_ham  = counts['ham']  / ham_emails
    ratio = p_word_given_spam / p_word_given_ham if p_word_given_ham > 0 else float('inf')
    indicator = 'Strong spam signal' if ratio > 2 else (' Ham signal' if ratio < 0.5 else '—')
    print(f"'{word}'   {p_word_given_spam:.3f}              {p_word_given_ham:.3f}              {indicator}")

---
## 3️⃣ Bayes' Theorem

### Concept
**Bayes' Theorem** lets us reverse conditional probabilities — we can go from $P(\text{word}|\text{spam})$ to $P(\text{spam}|\text{word})$:

$$P(\text{spam} \mid \text{word}) = \frac{P(\text{word} \mid \text{spam}) \cdot P(\text{spam})}{P(\text{word})}$$

In general:

$$P(A \mid B) = \frac{P(B \mid A) \cdot P(A)}{P(B)}$$

| Term | Name | Meaning |
|---|---|---|
| $P(\text{spam} \mid \text{word})$ | **Posterior** | What we want — probability of class given features |
| $P(\text{word} \mid \text{spam})$ | **Likelihood** | How probable the feature is given the class |
| $P(\text{spam})$ | **Prior** | Base rate of the class |
| $P(\text{word})$ | **Evidence** | How common the feature is overall |

In [ ]:
# Bayes' Theorem: what is P(spam | 'free' is in email)?

word = 'free'
n_spam_with_free = word_counts[word]['spam']
n_ham_with_free  = word_counts[word]['ham']

# Likelihood
P_free_given_spam = n_spam_with_free / spam_emails
P_free_given_ham  = n_ham_with_free  / ham_emails

# Evidence (total probability of 'free' appearing)
P_free = (n_spam_with_free + n_ham_with_free) / total_emails

# Bayes' Theorem
P_spam_given_free = (P_free_given_spam * P_spam) / P_free
P_ham_given_free  = (P_free_given_ham  * P_ham)  / P_free

print("=== Bayes' Theorem applied to spam detection ===")
print(f"\nP(spam)             = {P_spam:.3f}  ← prior")
print(f"P('free' | spam)    = {P_free_given_spam:.3f}  ← likelihood")
print(f"P('free')           = {P_free:.3f}  ← evidence")
print(f"\nP(spam | 'free')    = {P_spam_given_free:.3f}  ← posterior")
print(f"P(ham  | 'free')    = {P_ham_given_free:.3f}  ← posterior")
print(f"\nConclusion: An email containing 'free' has a {P_spam_given_free*100:.1f}% chance of being spam.")

In [ ]:
# Naive Bayes with MULTIPLE words (the 'naive' assumption: features are independent)
# P(spam | free, meeting) ∝ P(free|spam) * P(meeting|spam) * P(spam)

def naive_bayes_predict(words_in_email, word_counts, P_spam, P_ham, spam_emails, ham_emails):
    """Classify an email given a list of words in it."""
    log_prob_spam = np.log(P_spam)
    log_prob_ham  = np.log(P_ham)
    
    for word in words_in_email:
        if word in word_counts:
            # Laplace smoothing: add 1 to avoid zero probabilities
            p_word_spam = (word_counts[word]['spam'] + 1) / (spam_emails + len(word_counts))
            p_word_ham  = (word_counts[word]['ham']  + 1) / (ham_emails  + len(word_counts))
            log_prob_spam += np.log(p_word_spam)
            log_prob_ham  += np.log(p_word_ham)
    
    return 'spam' if log_prob_spam > log_prob_ham else 'ham'

# Test emails
test_emails = [
    (['free', 'free', 'the'],          'Expected: spam'),
    (['meeting', 'the'],               'Expected: ham'),
    (['free', 'meeting', 'the'],       'Expected: depends on weight'),
]

print("=== Naive Bayes Classifier ===")
for words, expectation in test_emails:
    prediction = naive_bayes_predict(words, word_counts, P_spam, P_ham, spam_emails, ham_emails)
    print(f"Email words: {', '.join(words):<35} → Prediction: {prediction:<6} ({expectation})")

---
## 4️⃣ Normal Distribution (Gaussian)

### Concept
Many real-world measurements follow a **Normal (Gaussian) distribution** — a symmetric bell curve.

$$f(x) = \frac{1}{\sigma \sqrt{2\pi}} \exp\left(-\frac{(x - \mu)^2}{2\sigma^2}\right)$$

- $\mu$ (mu) = mean — the center of the bell
- $\sigma$ (sigma) = standard deviation — the width of the bell

**ML connection:** **Gaussian Naive Bayes** uses the Normal distribution to model the likelihood $P(\text{feature value} \mid \text{class})$ when features are continuous (e.g., height, petal length).

In [ ]:
# Visualise Normal distributions for two classes
x = np.linspace(0, 15, 500)

# Petal length for two Iris species (hypothetical)
mu_setosa     = 1.5;  sigma_setosa     = 0.4
mu_versicolor = 4.3;  sigma_versicolor = 0.7

pdf_setosa     = stats.norm.pdf(x, mu_setosa,     sigma_setosa)
pdf_versicolor = stats.norm.pdf(x, mu_versicolor, sigma_versicolor)

plt.figure(figsize=(8, 4))
plt.plot(x, pdf_setosa,     'steelblue',  linewidth=2.5, label=f'Setosa (μ={mu_setosa}, σ={sigma_setosa})')
plt.plot(x, pdf_versicolor, 'tomato',     linewidth=2.5, label=f'Versicolor (μ={mu_versicolor}, σ={sigma_versicolor})')
plt.axvline(3.0, linestyle='--', color='gray', linewidth=1, label='New sample (petal_length=3.0)')
plt.xlabel('Petal length (cm)'); plt.ylabel('Probability Density')
plt.title('Gaussian Naive Bayes: P(feature | class) for each class')
plt.legend(); plt.grid(True, alpha=0.4); plt.tight_layout(); plt.show()

In [ ]:
# Gaussian Naive Bayes classification using the PDF
new_petal_length = 3.0

# Likelihood P(x | class) using the Normal PDF
likelihood_setosa     = stats.norm.pdf(new_petal_length, mu_setosa,     sigma_setosa)
likelihood_versicolor = stats.norm.pdf(new_petal_length, mu_versicolor, sigma_versicolor)

# Equal priors (50/50)
prior = 0.5

# Posterior ∝ likelihood × prior (ignoring the evidence denominator since it's the same)
posterior_setosa     = likelihood_setosa     * prior
posterior_versicolor = likelihood_versicolor * prior

# Normalise
total = posterior_setosa + posterior_versicolor
prob_setosa     = posterior_setosa     / total
prob_versicolor = posterior_versicolor / total

print(f"New sample: petal_length = {new_petal_length} cm")
print(f"\nP(petal={new_petal_length} | setosa)     = {likelihood_setosa:.4f}")
print(f"P(petal={new_petal_length} | versicolor) = {likelihood_versicolor:.4f}")
print(f"\nNormalised posteriors:")
print(f"  P(setosa | petal={new_petal_length})     = {prob_setosa:.3f} ({prob_setosa*100:.1f}%)")
print(f"  P(versicolor | petal={new_petal_length}) = {prob_versicolor:.3f} ({prob_versicolor*100:.1f}%)")
print(f"\nPrediction: {'setosa' if prob_setosa > prob_versicolor else 'versicolor'}")

---
## 5️⃣ PDF and CDF

### Concept
- **PDF (Probability Density Function):** $f(x)$ — the density at a point $x$. Area under the curve = 1.
- **CDF (Cumulative Distribution Function):** $F(x) = P(X \leq x)$ — probability of being at or below $x$.

**ML connection:** 
- The PDF gives us the likelihood value Gaussian Naive Bayes uses
- CDFs are used in threshold-based decisions and confidence intervals

In [ ]:
# PDF vs CDF
mu, sigma = 0, 1  # standard normal
x = np.linspace(-4, 4, 300)

pdf = stats.norm.pdf(x, mu, sigma)
cdf = stats.norm.cdf(x, mu, sigma)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# PDF
axes[0].plot(x, pdf, 'steelblue', linewidth=2.5)
x_fill = np.linspace(-1, 1, 200)
axes[0].fill_between(x_fill, stats.norm.pdf(x_fill), alpha=0.3, color='steelblue', label='P(-1 ≤ X ≤ 1) ≈ 68%')
axes[0].set_title('PDF — Probability Density Function')
axes[0].set_xlabel('x'); axes[0].set_ylabel('f(x) — density')
axes[0].legend(); axes[0].grid(True, alpha=0.4)

# CDF
axes[1].plot(x, cdf, 'tomato', linewidth=2.5)
axes[1].axhline(0.5, linestyle='--', color='gray', linewidth=1, label='50th percentile (median=0)')
axes[1].axvline(0, linestyle='--', color='gray', linewidth=1)
axes[1].set_title('CDF — Cumulative Distribution Function')
axes[1].set_xlabel('x'); axes[1].set_ylabel('F(x) = P(X ≤ x)')
axes[1].legend(); axes[1].grid(True, alpha=0.4)

plt.suptitle('Standard Normal Distribution: PDF and CDF', fontsize=11)
plt.tight_layout(); plt.show()

print(f"P(X ≤ 0):   CDF(0)   = {stats.norm.cdf(0):.3f}   (50% — by symmetry)")
print(f"P(X ≤ 1.96): CDF(1.96) = {stats.norm.cdf(1.96):.3f}  (95th percentile)")
print(f"P(-1 ≤ X ≤ 1):          = {stats.norm.cdf(1) - stats.norm.cdf(-1):.3f}  (68% rule)")

---
## 6️⃣ sklearn GaussianNB — Connecting everything

Now let's use `sklearn`'s Gaussian Naive Bayes on the Iris dataset and confirm that every number it computes is the probability math we just covered.

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Load Iris
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train
gnb = GaussianNB()
gnb.fit(X_train, y_train)

# Predict
y_pred = gnb.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

In [ ]:
# Show what sklearn learned — these are exactly the mu and sigma for each class
print("What GaussianNB learned (μ and σ per feature per class):")
print(f"{'':20} {'setosa':>12} {'versicolor':>12} {'virginica':>12}")
print("-" * 60)

for i, feat in enumerate(iris.feature_names):
    means = gnb.theta_[:, i]   # mean per class
    stds  = np.sqrt(gnb.var_[:, i])  # std per class
    print(f"{feat[:19]:20} {means[0]:>6.2f}±{stds[0]:.2f}   {means[1]:>6.2f}±{stds[1]:.2f}   {means[2]:>6.2f}±{stds[2]:.2f}")

print("\nThese μ and σ values define the Normal distributions used to compute P(feature | class).")
print("That is exactly the Gaussian PDF we coded from scratch above!")

---
## 🔗 Explore further in this repo

You now understand the probability theory behind Naive Bayes — every operation in the `../../naive_bayes_implementation.ipynb` folder in this repo maps to a concept from this notebook:

- `P(spam)` → **Prior**
- `P(word | spam)` → **Likelihood** (counted from training data)
- Bayes' Theorem → **Posterior** (what the classifier outputs)
- `GaussianNB` → **Normal PDF** as the likelihood for continuous features

**Previous notebook:** [02_Calculus_for_ML.ipynb](02_Calculus_for_ML.ipynb)

---

## 📝 Summary

| Concept | Formula | ML use |
|---|---|---|
| Prior | $P(C)$ | Base rate of each class |
| Likelihood | $P(x \mid C)$ | How probable is this feature given the class |
| Bayes' Theorem | $P(C \mid x) = \frac{P(x\mid C)P(C)}{P(x)}$ | Naive Bayes classifier |
| Normal PDF | $f(x) = \frac{1}{\sigma\sqrt{2\pi}}e^{-\frac{(x-\mu)^2}{2\sigma^2}}$ | Gaussian Naive Bayes likelihood |
| CDF | $F(x) = P(X \leq x)$ | Thresholds, confidence intervals |

---
📺 **Deepen your intuition:** [StatQuest with Josh Starmer](https://www.youtube.com/c/joshstarmer)